# Notebook 03: Topic Clustering

## Goal

Discover latent topic structure in MedQA questions using BERTopic.

The classifier in notebook 02 assigns a broad specialty label (19 classes) to each question.
Clustering goes one level deeper: it finds fine-grained topic groups *within* the question bank
without using any labels, driven entirely by the question text. These topic assignments are
used at RAG inference time to retrieve topically similar questions as few-shot context.

## Research Question Re-Test
A2 found TF-IDF + GMM/EM (kappa = 0.418) as the frequency-based champion and
PubMedBERT + Spectral Clustering as the state-of-the-art champion on a highly overlapping
corpus (pairwise similarity ~0.95). With MedMCQA showing 0.72 mean inter-category similarity,
do the A2 champions hold, or does the lower overlap change the ranking?

## Evaluation protocol (A2 Methodology)
- Primary metric   : Cohen's kappa (topic vs specialty label agreement)
- Secondary metrics: Silhouette score, Topic Coherence C_v
- A2 baselines     : TF-IDF + GMM (kappa 0.418), Embeddings + Spectral
- Primary method   : BERTopic (HDBSCAN-based, automatic topic count)

<div class="alert alert-block alert-info">
<b style="font-size: 1.2em;">ⓘ Note</b>

BERTopic determines K automatically from the data via HDBSCAN. The only parameter we set is `min_cluster_size` which controls granularity.
</div>

## Output
```
models/clustering/
  bertopic_model/              # saved BERTopic model
  medqa_with_topics.parquet    # MedQA with topic assignments
  config.json                  # evaluation results
```

## 0. Setup

In [ ]:
# Imports
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import SpectralClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import cohen_kappa_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import LabelEncoder

from src.cluster import (
    evaluate_topics,
    format_eval,
    get_topic_words,
    topic_specialty_alignment,
)
from src.data import REPO_ROOT
from src.vectorstore import BIOMEDICAL_MODEL, embed_texts, load_embedding_model

warnings.filterwarnings('ignore')

CLUSTERING_DIR = REPO_ROOT / 'models' / 'clustering'
CLUSTERING_DIR.mkdir(parents=True, exist_ok=True)
CLASSIFIER_DIR = REPO_ROOT / 'models' / 'classifier'

print(f'Repo root     : {REPO_ROOT}')
print(f'Clustering dir: {CLUSTERING_DIR}')

In [ ]:
# ── COLAB ONLY — skip if running locally ─────────────────────────────────────
# Run once per Colab session. Requires GITHUB_REPO_URL in Colab Secrets.
# Also restores classifier outputs from Drive (needed for specialty labels).
# -----------------------------------------------------------------------------

import os, shutil
from pathlib import Path
from google.colab import drive, userdata

repo_url = userdata.get('GITHUB_REPO_URL')
if not Path('/content/emma').exists():
    !git clone {repo_url}
os.chdir('/content/emma')
!pip install -e . -q
!pip install faiss-cpu sentence-transformers bertopic gensim umap-learn hdbscan -q

drive.mount('/content/drive')
DRIVE_CLF = Path('/content/drive/MyDrive/emma/models/classifier')
DRIVE_OUT = Path('/content/drive/MyDrive/emma/models/clustering')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Restore classifier outputs from Drive
local_clf = Path('/content/emma/models/classifier')
local_clf.mkdir(parents=True, exist_ok=True)
for f in DRIVE_CLF.glob('*'):
    shutil.copy(f, local_clf / f.name)

# Re-import
import json, warnings
import matplotlib.pyplot as plt, numpy as np, pandas as pd, seaborn as sns
from sklearn.cluster import SpectralClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import cohen_kappa_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import LabelEncoder
from src.cluster import evaluate_topics, format_eval, get_topic_words, topic_specialty_alignment
from src.data import REPO_ROOT
from src.vectorstore import BIOMEDICAL_MODEL, embed_texts, load_embedding_model
warnings.filterwarnings('ignore')
CLUSTERING_DIR = REPO_ROOT / 'models' / 'clustering'
CLUSTERING_DIR.mkdir(parents=True, exist_ok=True)
CLASSIFIER_DIR = REPO_ROOT / 'models' / 'classifier'
print('Colab setup complete.')

## 1. Data Loading

Load MedQA with specialty labels from notebook 02.
These labels serve as ground truth for Cohen's kappa evaluation.

In [ ]:
medqa_path = CLASSIFIER_DIR / 'medqa_with_specialty.parquet'
if not medqa_path.exists():
    raise FileNotFoundError(
        f'{medqa_path} not found.\n'
        'Run notebook 02_classification.ipynb first.'
    )

medqa = pd.read_parquet(medqa_path)
docs        = medqa['question'].tolist()
specialties = medqa['specialty'].tolist()

le = LabelEncoder()
y  = le.fit_transform(specialties)
K  = medqa['specialty'].nunique()   # data-driven K for baselines

print(f'Questions   : {len(docs):,}')
print(f'Specialties : {K}  (used as K for GMM/Spectral baselines)')
display(medqa['specialty'].value_counts().reset_index())

## 2. Pre-Compute Embeddings

All three methods use the same embedding space for a fair comparison.
Pre-computing here avoids re-embedding for each method.

In [ ]:
# USE_FAST_MODEL=True for local CPU (~10 min); False for Qwen3 on GPU
USE_FAST_MODEL = True
EMB_MODEL = 'all-MiniLM-L6-v2' if USE_FAST_MODEL else BIOMEDICAL_MODEL

print(f'Embedding model: {EMB_MODEL}')
print(f'Documents      : {len(docs):,}')
emb_model  = load_embedding_model(EMB_MODEL)
embeddings = embed_texts(docs, emb_model, batch_size=32)
print(f'Embedding matrix: {embeddings.shape}')

## 3. A2 Baseline 1: TF-IDF + GMM

The A2 frequency-based champion (kappa = 0.418 on PubMed AI corpus).
Reproduced as a lower-bound reference point.

<div class="alert alert-block alert-info">
<b style="font-size: 1.2em;">ⓘ Note</b>

K is set to the number of unique specialties (19, derived from the data) so kappa is comparable across methods. BERTopic determines K automatically in section 5.
</div>

In [ ]:
print('Fitting TF-IDF...')
tfidf   = TfidfVectorizer(max_features=20_000, sublinear_tf=True, min_df=2)
X_tfidf = tfidf.fit_transform(docs).toarray()   # GMM needs dense
print(f'TF-IDF matrix: {X_tfidf.shape}')

print(f'\nFitting GMM (K={K}, covariance=full)...')
gmm = GaussianMixture(
    n_components=K, covariance_type='full',
    max_iter=200, random_state=42, verbose=1,
)
gmm.fit(X_tfidf)
gmm_labels = gmm.predict(X_tfidf)

res_gmm = evaluate_topics(
    topic_labels=gmm_labels,
    specialty_labels=specialties,
)
print('\n' + format_eval(res_gmm, 'TF-IDF + GMM (A2 baseline)'))

## 4. A2 Baseline 2: Embeddings + Spectral Clustering

The A2 state-of-the-art champion. Uses the pre-computed embeddings.

<div class="alert alert-block alert-warning">
<b style="font-size: 1.2em;">⚠ Warning</b>

Spectral Clustering builds an $N \times N$ affinity matrix.
At 12,723 questions this requires ~1.2 GB RAM. Runtime is ~2 min on Colab T4.
</div>

In [ ]:
print(f'Fitting Spectral Clustering (K={K})...')
spectral = SpectralClustering(
    n_clusters=K, affinity='rbf',
    random_state=42, n_jobs=-1, verbose=True,
)
spectral_labels = spectral.fit_predict(embeddings)

res_spectral = evaluate_topics(
    topic_labels=spectral_labels,
    specialty_labels=specialties,
    embeddings=embeddings,
)
print('\n' + format_eval(res_spectral, 'Embeddings + Spectral (A2 baseline)'))

## 5. BERTopic

BERTopic uses HDBSCAN to determine K automatically from the data, so there is no topic count to specify. Topics are represented by c-TF-IDF term weights.

The pre-computed embeddings are passed directly so BERTopic uses the same embedding space as the Spectral baseline. K emerges from `min_cluster_size`, the only meaningful hyperparameter here.

In [ ]:
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

umap_model = UMAP(
    n_neighbors=15, n_components=5,
    min_dist=0.0, metric='cosine', random_state=42,
)

# min_cluster_size=50 ~ 0.4% of corpus; keeps topics meaningful without over-splitting
hdbscan_model = HDBSCAN(
    min_cluster_size=50, min_samples=10,
    metric='euclidean', cluster_selection_method='eom',
    prediction_data=True,
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    embedding_model=EMB_MODEL,
    language='english',
    calculate_probabilities=False,
    verbose=True,
)

print('Fitting BERTopic...')
topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
topics = list(topics)

n_topics   = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = topics.count(-1)
print(f'\nTopics discovered : {n_topics}')
print(f'Outliers (topic -1): {n_outliers:,}  ({100*n_outliers/len(topics):.1f}%)')

In [ ]:
# Topic info
topic_info = topic_model.get_topic_info()
print(f'Topic info ({len(topic_info)} rows including outlier -1):')
display(topic_info.head(20))

In [ ]:
# Evaluate
topic_words_list = get_topic_words(topic_model, n_words=10)

res_bertopic = evaluate_topics(
    topic_labels=topics,
    specialty_labels=specialties,
    embeddings=embeddings,
    topic_words=topic_words_list,
    texts=docs,
)
print(format_eval(res_bertopic, 'BERTopic (automatic K)'))

## 6. Results Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Method':         'TF-IDF + GMM',
        'Type':           'A2 baseline',
        'K':              K,
        'Outliers':       '0 (0%)',
        "Cohen's kappa":  res_gmm['kappa'],
        'Silhouette':     res_gmm['silhouette'],
        'C_v':            res_gmm['coherence_cv'],
        'A2 kappa (ref)': 0.418,
    },
    {
        'Method':         'Embeddings + Spectral',
        'Type':           'A2 baseline',
        'K':              K,
        'Outliers':       '0 (0%)',
        "Cohen's kappa":  res_spectral['kappa'],
        'Silhouette':     res_spectral['silhouette'],
        'C_v':            res_spectral['coherence_cv'],
        'A2 kappa (ref)': 'N/A',
    },
    {
        'Method':         'BERTopic',
        'Type':           'Primary method',
        'K':              res_bertopic['n_topics'],
        'Outliers':       f"{res_bertopic['n_outliers']} ({res_bertopic['outlier_pct']}%)",
        "Cohen's kappa":  res_bertopic['kappa'],
        'Silhouette':     res_bertopic['silhouette'],
        'C_v':            res_bertopic['coherence_cv'],
        'A2 kappa (ref)': 'N/A',
    },
])
display(comparison_df)

## 7. BERTopic Visualisations

In [ ]:
# Top-word barchart for 8 largest topics
topic_model.visualize_barchart(top_n_topics=8, n_words=8)

In [ ]:
# Topic similarity heatmap
n_show = min(30, res_bertopic['n_topics'])
topic_model.visualize_heatmap(n_clusters=n_show)

In [ ]:
# Topic-specialty alignment — qualitative validation
alignment_df = topic_specialty_alignment(topics, specialties, topic_model)
print('Topic-specialty alignment (top 20 topics by size):')
display(alignment_df.head(20))

In [ ]:
# Specialty purity distribution
fig, ax = plt.subplots(figsize=(8, 3))
alignment_df['specialty_pct'].plot(
    kind='hist', bins=20, ax=ax, color='steelblue', edgecolor='white'
)
med = alignment_df['specialty_pct'].median()
ax.axvline(med, color='red', linestyle='--', label=f'Median {med:.1f}%')
ax.set_xlabel('Dominant specialty % per topic')
ax.set_ylabel('Count')
ax.set_title('Topic specialty purity — BERTopic')
ax.legend()
plt.tight_layout()
plt.show()
print('Higher purity -> topics align well with known specialties.')

## 8. Save Outputs

In [ ]:
# Attach topic assignments
medqa['topic'] = topics

out_path = CLUSTERING_DIR / 'medqa_with_topics.parquet'
medqa[['question', 'options', 'answer_idx', 'answer', 'split', 'specialty', 'topic']].to_parquet(
    out_path, index=False
)
print(f'Saved -> {out_path}  ({out_path.stat().st_size/1024:.0f} KB)')

# BERTopic model
model_path = CLUSTERING_DIR / 'bertopic_model'
topic_model.save(str(model_path), serialization='pickle')
print(f'Saved -> {model_path}')

# Config
config = {
    'embedding_model': EMB_MODEL,
    'n_questions':     len(medqa),
    'n_specialties':   K,
    'bertopic':  {
        'n_topics':     res_bertopic['n_topics'],
        'n_outliers':   res_bertopic['n_outliers'],
        'outlier_pct':  res_bertopic['outlier_pct'],
        'kappa':        res_bertopic['kappa'],
        'silhouette':   res_bertopic['silhouette'],
        'coherence_cv': res_bertopic['coherence_cv'],
    },
    'tfidf_gmm': {'kappa': res_gmm['kappa'], 'a2_baseline': 0.418},
    'spectral':  {'kappa': res_spectral['kappa'], 'silhouette': res_spectral['silhouette']},
}
with open(CLUSTERING_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f'Saved -> {CLUSTERING_DIR / "config.json"}')

In [ ]:
# ── COLAB ONLY: copy outputs to Drive ─────────────────────────────────────────
import shutil
for f in CLUSTERING_DIR.glob('*'):
    if f.is_file():
        shutil.copy(f, DRIVE_OUT / f.name)
        print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB) -> Drive')
if (CLUSTERING_DIR / 'bertopic_model').exists():
    shutil.copytree(CLUSTERING_DIR / 'bertopic_model', DRIVE_OUT / 'bertopic_model', dirs_exist_ok=True)
    print('  bertopic_model/ -> Drive')
print(f'Done. Drive path: {DRIVE_OUT}')

## 9. Summary

In [ ]:
config = json.loads((CLUSTERING_DIR / 'config.json').read_text())
bt = config['bertopic']
gm = config['tfidf_gmm']
sp = config['spectral']

summary_df = pd.DataFrame([
    {'Item': 'Questions clustered',       'Value': f"{config['n_questions']:,}"},
    {'Item': 'Specialties (ground truth)', 'Value': str(config['n_specialties'])},
    {'Item': 'Embedding model',           'Value': config['embedding_model']},
    {'Item': 'BERTopic K (auto)',         'Value': str(bt['n_topics'])},
    {'Item': 'BERTopic outliers',         'Value': f"{bt['n_outliers']:,}  ({bt['outlier_pct']}%)"},
    {'Item': 'BERTopic kappa',            'Value': str(bt['kappa'])},
    {'Item': 'BERTopic silhouette',       'Value': str(bt['silhouette'])},
    {'Item': 'BERTopic C_v coherence',    'Value': str(bt['coherence_cv'])},
    {'Item': 'TF-IDF + GMM kappa',        'Value': f"{gm['kappa']}  (A2 ref: {gm['a2_baseline']})"},
    {'Item': 'Spectral kappa',            'Value': str(sp['kappa'])},
    {'Item': 'Model saved',               'Value': str(CLUSTERING_DIR / 'bertopic_model')},
])

display(summary_df)
print('\nNext step: 04_rag_pipeline.ipynb — end-to-end RAG with NER + retrieval + LLM.')